# 04 — Model Evaluation
**Project:** QLoRA Fine-Tuning for Functional Job Skill Classification  
**Model:** Microsoft Phi-3 Mini (`microsoft/Phi-3-mini-4k-instruct`)  
**Group 20:** Michael Dong, Haolin Chen

---

This notebook evaluates the fine-tuned Phi-3 Mini model against the held-out test set and produces:
- **Baseline vs. fine-tuned comparison** (Exact Match Accuracy, F1, Precision, Recall)
- **Per-class classification report** for the fine-tuned model
- **Perplexity** on the test split
- **Confusion matrix** for top-N skill categories
- **Qualitative sample predictions** side-by-side

Run this notebook after `03_phi3_qlora_training.ipynb` has saved the LoRA adapters.

## 0. Install Dependencies

In [ ]:
# Run once on Kaggle / Colab — skip if already installed
!pip install -q transformers peft bitsandbytes accelerate \
                scikit-learn pandas matplotlib seaborn tqdm

## 1. Configuration

Update `ADAPTER_PATH` and `TEST_DATA_PATH` to match the paths used in notebook 03.

In [ ]:
import os
import json
import math
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_MODEL_ID   = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER_PATH    = "./lora_adapters"          # saved by notebook 03
TEST_DATA_PATH  = "./test_dataset.csv"       # saved by notebook 02

# ── Generation config ────────────────────────────────────────────────────────
MAX_INPUT_LEN   = 512    # tokens — match training
MAX_NEW_TOKENS  = 20     # skill_name is short; keep inference fast

# ── Evaluation config ────────────────────────────────────────────────────────
TOP_N_CLASSES   = 15     # classes shown in confusion matrix
SAMPLE_N        = 10     # qualitative examples to print
RANDOM_SEED     = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Adapter path exists: {os.path.exists(ADAPTER_PATH)}")
print(f"Test data exists:    {os.path.exists(TEST_DATA_PATH)}")

## 2. Load Test Dataset

In [ ]:
df_test = pd.read_csv(TEST_DATA_PATH)

# Expected columns: 'instruction', 'response' (the ground-truth skill_name)
# Adjust column names below if notebook 02 used different names.
INSTRUCTION_COL = "instruction"
LABEL_COL       = "response"        # ground-truth skill_name

assert INSTRUCTION_COL in df_test.columns, f"Missing column: {INSTRUCTION_COL}"
assert LABEL_COL       in df_test.columns, f"Missing column: {LABEL_COL}"

df_test = df_test[[INSTRUCTION_COL, LABEL_COL]].dropna().reset_index(drop=True)

print(f"Test samples: {len(df_test):,}")
print(f"Unique skill classes: {df_test[LABEL_COL].nunique()}")
df_test.head(3)

## 3. Load Models

We load two versions for comparison:
- **Baseline** — Phi-3 Mini with no fine-tuning
- **Fine-tuned** — Phi-3 Mini + QLoRA adapters from notebook 03

In [ ]:
# 4-bit quantization config — mirrors training setup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"    # required for causal LM batch inference

print("Loading base model (baseline)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()
print("Base model loaded.")

print("Loading fine-tuned model (base + LoRA adapters)...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print("Fine-tuned model loaded.")

## 4. Inference Helper

Given a model and a list of instruction strings, returns predicted `skill_name` strings.

In [ ]:
def predict_skill(model, instructions: list[str], batch_size: int = 8) -> list[str]:
    """
    Run batched inference and extract the predicted skill_name from each output.
    The model is expected to generate the skill label immediately after the
    instruction text; we strip the prompt and take the first non-empty line.
    """
    predictions = []

    for i in tqdm(range(0, len(instructions), batch_size), desc="Inference"):
        batch = instructions[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy — deterministic for evaluation
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens (not the prompt)
        input_len = inputs["input_ids"].shape[1]
        for output in outputs:
            new_tokens = output[input_len:]
            decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            # Take first line only — skill names are single phrases
            first_line = decoded.split("\n")[0].strip()
            predictions.append(first_line if first_line else "[empty]")

    return predictions


def normalize(text: str) -> str:
    """Lowercase + strip for case-insensitive exact match."""
    return str(text).lower().strip()

## 5. Generate Predictions

In [ ]:
instructions  = df_test[INSTRUCTION_COL].tolist()
ground_truths = df_test[LABEL_COL].tolist()

print("=" * 50)
print("Running baseline (no fine-tuning) inference...")
baseline_preds = predict_skill(base_model, instructions)

print("=" * 50)
print("Running fine-tuned model inference...")
ft_preds = predict_skill(ft_model, instructions)

# Save raw predictions for reproducibility
results_df = pd.DataFrame({
    "instruction":     instructions,
    "ground_truth":    ground_truths,
    "baseline_pred":   baseline_preds,
    "finetuned_pred":  ft_preds,
})
results_df.to_csv("evaluation_results.csv", index=False)
print("Predictions saved to evaluation_results.csv")

## 6. Exact Match Accuracy — Baseline vs. Fine-tuned

In [ ]:
gt_norm   = [normalize(g) for g in ground_truths]
base_norm = [normalize(p) for p in baseline_preds]
ft_norm   = [normalize(p) for p in ft_preds]

base_acc = accuracy_score(gt_norm, base_norm)
ft_acc   = accuracy_score(gt_norm, ft_norm)

print(f"Exact Match Accuracy")
print(f"  Baseline (no fine-tuning): {base_acc:.4f}  ({base_acc*100:.2f}%)")
print(f"  Fine-tuned (QLoRA):        {ft_acc:.4f}  ({ft_acc*100:.2f}%)")
print(f"  Improvement:               +{(ft_acc - base_acc)*100:.2f} pp")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Baseline\n(Phi-3 Mini)", "Fine-tuned\n(QLoRA)"],
    [base_acc * 100, ft_acc * 100],
    color=["#90B8D4", "#1F3864"],
    width=0.45,
    edgecolor="white",
)
for bar, val in zip(bars, [base_acc, ft_acc]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val*100:.2f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
ax.set_ylabel("Exact Match Accuracy (%)", fontsize=11)
ax.set_title("Baseline vs. Fine-tuned — Exact Match Accuracy", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(ft_acc * 100 * 1.15, 10))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
sns.despine()
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()
print("Saved: accuracy_comparison.png")

## 7. Precision, Recall & F1 — Fine-tuned Model

We compute micro, macro, and weighted averages to handle class imbalance in the LinkedIn dataset.

In [ ]:
# Only include labels seen in ground truth to avoid sklearn warnings
labels = sorted(set(gt_norm))

metrics = {}
for avg in ["micro", "macro", "weighted"]:
    metrics[avg] = {
        "precision": precision_score(gt_norm, ft_norm, average=avg, labels=labels, zero_division=0),
        "recall":    recall_score(gt_norm, ft_norm, average=avg, labels=labels, zero_division=0),
        "f1":        f1_score(gt_norm, ft_norm, average=avg, labels=labels, zero_division=0),
    }

metrics_df = pd.DataFrame(metrics).T * 100  # convert to percentage
print("Fine-tuned Model — Precision / Recall / F1 (%)")
print(metrics_df.round(2).to_string())

# Grouped bar chart
x = np.arange(len(metrics))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#4472C4", "#ED7D31", "#A9D18E"]
for i, metric in enumerate(["precision", "recall", "f1"]):
    vals = [metrics[avg][metric] * 100 for avg in ["micro", "macro", "weighted"]]
    rects = ax.bar(x + i * width, vals, width, label=metric.capitalize(), color=colors[i])
    for rect, val in zip(rects, vals):
        ax.text(
            rect.get_x() + rect.get_width() / 2,
            rect.get_height() + 0.3,
            f"{val:.1f}",
            ha="center", va="bottom", fontsize=8
        )

ax.set_xticks(x + width)
ax.set_xticklabels(["Micro", "Macro", "Weighted"], fontsize=11)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("Fine-tuned Model — Precision, Recall, F1", fontsize=12, fontweight="bold")
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig("prf1_metrics.png", dpi=150)
plt.show()
print("Saved: prf1_metrics.png")

## 8. Per-Class Classification Report

In [ ]:
report = classification_report(
    gt_norm, ft_norm,
    labels=labels,
    zero_division=0,
    output_dict=True
)
report_df = pd.DataFrame(report).T

# Show top 20 classes by support (most common in test set)
top_classes = (
    report_df
    .iloc[:-3]          # exclude micro/macro/weighted summary rows
    .sort_values("support", ascending=False)
    .head(20)
)

print("Per-class report (top 20 by support):")
print(top_classes[["precision", "recall", "f1-score", "support"]].round(3).to_string())

## 9. Confusion Matrix — Top N Classes

In [ ]:
# Select top-N classes by frequency in ground truth
top_n_labels = (
    pd.Series(gt_norm)
    .value_counts()
    .head(TOP_N_CLASSES)
    .index.tolist()
)

# Filter predictions to only these classes
mask = pd.Series(gt_norm).isin(top_n_labels)
gt_top   = pd.Series(gt_norm)[mask].tolist()
pred_top = pd.Series(ft_norm)[mask].tolist()
# Map out-of-top predictions to "other"
pred_top = [p if p in top_n_labels else "[other]" for p in pred_top]
cm_labels = top_n_labels + (["[other]"] if "[other]" in pred_top else [])

cm = confusion_matrix(gt_top, pred_top, labels=cm_labels)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True, fmt="d",
    xticklabels=cm_labels, yticklabels=cm_labels,
    cmap="Blues",
    linewidths=0.5,
    ax=ax
)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
ax.set_title(f"Confusion Matrix — Top {TOP_N_CLASSES} Skill Classes (Fine-tuned)",
             fontsize=13, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

## 10. Perplexity on Test Set

Perplexity measures how well the model predicts the test sequences as a language model.  
Lower is better. We compute it for both models to quantify how much fine-tuning shifted the distribution.

In [ ]:
def compute_perplexity(model, texts: list[str], batch_size: int = 4) -> float:
    """
    Compute average per-token perplexity over a list of texts.
    Uses the causal LM loss (cross-entropy over next-token predictions).
    """
    total_loss  = 0.0
    total_tokens = 0

    for i in tqdm(range(0, len(texts), batch_size), desc="Perplexity"):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(DEVICE)

        labels = enc["input_ids"].clone()
        # Mask padding tokens in the loss
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            out = model(**enc, labels=labels)

        # out.loss is mean cross-entropy per non-masked token
        n_tokens = (labels != -100).sum().item()
        total_loss   += out.loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)


# Use full instruction+response as the text for perplexity
full_texts = [
    f"{instr}\n{resp}"
    for instr, resp in zip(instructions, ground_truths)
]

print("Computing baseline perplexity...")
base_ppl = compute_perplexity(base_model, full_texts)

print("Computing fine-tuned perplexity...")
ft_ppl = compute_perplexity(ft_model, full_texts)

print(f"\nPerplexity Results")
print(f"  Baseline  : {base_ppl:.4f}")
print(f"  Fine-tuned: {ft_ppl:.4f}")
print(f"  Reduction : {base_ppl - ft_ppl:.4f}  ({(1 - ft_ppl/base_ppl)*100:.1f}% lower)")

## 11. Summary Table

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Exact Match Accuracy",
        "Precision (weighted)",
        "Recall (weighted)",
        "F1 (weighted)",
        "Perplexity",
    ],
    "Baseline": [
        f"{base_acc*100:.2f}%",
        "—",
        "—",
        "—",
        f"{base_ppl:.2f}",
    ],
    "Fine-tuned (QLoRA)": [
        f"{ft_acc*100:.2f}%",
        f"{metrics['weighted']['precision']*100:.2f}%",
        f"{metrics['weighted']['recall']*100:.2f}%",
        f"{metrics['weighted']['f1']*100:.2f}%",
        f"{ft_ppl:.2f}",
    ],
    "Direction": ["↑ higher is better"] * 4 + ["↓ lower is better"],
})

print("\n=== EVALUATION SUMMARY ===")
print(summary.to_string(index=False))
summary.to_csv("evaluation_summary.csv", index=False)
print("\nSaved: evaluation_summary.csv")

## 12. Qualitative Examples — Side-by-side Predictions

In [ ]:
np.random.seed(RANDOM_SEED)
sample_idx = np.random.choice(len(results_df), size=SAMPLE_N, replace=False)
sample_df  = results_df.iloc[sample_idx].reset_index(drop=True)

print("=" * 80)
print(f"QUALITATIVE SAMPLE PREDICTIONS  (n={SAMPLE_N})")
print("=" * 80)

for _, row in sample_df.iterrows():
    # Truncate instruction for readability
    instr_preview = str(row[INSTRUCTION_COL])[:200].replace("\n", " ") + "..."
    gt   = str(row["ground_truth"])
    base = str(row["baseline_pred"])
    ft   = str(row["finetuned_pred"])

    base_correct = "✓" if normalize(base) == normalize(gt) else "✗"
    ft_correct   = "✓" if normalize(ft)   == normalize(gt) else "✗"

    print(f"\nInstruction (truncated):")
    print(f"  {instr_preview}")
    print(f"  Ground truth  : {gt}")
    print(f"  Baseline pred : {base}  {base_correct}")
    print(f"  Fine-tuned    : {ft}  {ft_correct}")
    print("-" * 60)

## 13. Error Analysis — Most Common Misclassifications

In [ ]:
errors_df = results_df[
    results_df.apply(
        lambda r: normalize(r["finetuned_pred"]) != normalize(r["ground_truth"]), axis=1
    )
].copy()

errors_df["error_pair"] = (
    errors_df["ground_truth"].str.lower().str.strip()
    + " → "
    + errors_df["finetuned_pred"].str.lower().str.strip()
)

top_errors = errors_df["error_pair"].value_counts().head(15)

print(f"Total errors: {len(errors_df)} / {len(results_df)}  "
      f"({len(errors_df)/len(results_df)*100:.1f}% error rate)")
print("\nTop 15 most frequent misclassifications (true → predicted):")
print(top_errors.to_string())

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
top_errors.sort_values().plot(
    kind="barh", ax=ax, color="#C9D9EC", edgecolor="#1F3864"
)
ax.set_xlabel("Count", fontsize=11)
ax.set_title("Top Misclassification Pairs (true skill → predicted skill)",
             fontsize=12, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.savefig("error_analysis.png", dpi=150)
plt.show()
print("Saved: error_analysis.png")

## 14. Save All Outputs

Collect all generated files for the final report and GitHub submission.

In [ ]:
output_files = [
    "evaluation_results.csv",
    "evaluation_summary.csv",
    "accuracy_comparison.png",
    "prf1_metrics.png",
    "confusion_matrix.png",
    "error_analysis.png",
]

print("Generated evaluation outputs:")
for f in output_files:
    exists = os.path.exists(f)
    print(f"  {'✓' if exists else '✗'}  {f}")

print("\nEvaluation complete. Move outputs to experiments/results/ in the repo.")

---

## References

- Abdin, M., et al. (2024). *Phi-3 Technical Report*. Microsoft / arXiv.
- Dettmers, T., et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*. arXiv:2305.14314.
- Hu, E., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.
- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12.
- [Hugging Face PEFT Documentation](https://huggingface.co/docs/peft)